<a href="https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page for one client on one report date.

For my lane, I will use March 2026 as the development and verification time window. I will use the March data to understand page performance and build features for the content refresh opportunity lane.

June 2026 will not be used for development because it is the final month and should be kept as a sealed test period.

In [29]:
%pip -q install duckdb

import duckdb

con = duckdb.connect()

print("DuckDB is ready.")

DuckDB is ready.


In [30]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("HF_TOKEN found successfully.")
else:
    print("HF_TOKEN was not found. Check Colab Secrets.")

HF_TOKEN found successfully.


In [31]:
import os

os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face token is ready for the warehouse connection.")

Hugging Face token is ready for the warehouse connection.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

I will use the following observed performance fields as candidate features:

* `gsc_impressions` — observed Google Search Console impressions.
* `gsc_clicks` — observed Google Search Console clicks.
* `gsc_avg_position` — observed average search position.
* `ga4_sessions` — observed GA4 sessions when GA4 data is available.
* CTR — calculated as `gsc_clicks / gsc_impressions` when impressions are greater than zero.

These features describe observed page performance and can be available before a future decision.

### Label

The label will represent a future observed page-performance outcome used to rank pages for the content refresh opportunity.

The future outcome must not be included as an input feature.

### Context

The following fields are context fields:

* `client_hash_id` — identifies the client group.
* `content_hash_id` — identifies the content page.
* `report_date` — identifies the observation date.
* `month` — identifies the data month.

These fields help identify, group, join, and split the observations but are not predictive features.

### Excluded

I will exclude:

* Future performance information, because it would cause leakage.
* Any field directly derived from the label, because it would give the model information about the answer.
* `client_hash_id` and `content_hash_id` as model features, because they are identifiers rather than meaningful performance signals.
* June 2026 during development, because it should remain a sealed test period.

I will also treat GA4 metrics carefully because `ga4_data_available` indicates whether GA4 data is available for that observation.


In [32]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ctr"
]

context = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month"
]

excluded = [
    "future performance information",
    "label-derived fields",
    "client_hash_id as a model feature",
    "content_hash_id as a model feature",
    "June 2026 during development"
]

print("Candidate features:", features)
print("Context fields:", context)
print("Excluded:", excluded)

Candidate features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr']
Context fields: ['client_hash_id', 'content_hash_id', 'report_date', 'month']
Excluded: ['future performance information', 'label-derived fields', 'client_hash_id as a model feature', 'content_hash_id as a model feature', 'June 2026 during development']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("Hugging Face authentication is ready.")

Hugging Face authentication is ready.


In [34]:
schema = con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [35]:
query1 = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS number_of_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
"""

result1 = con.sql(query1).df()

print("Query 1: Grain check")
display(result1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: Grain check


,client_hash_id,content_hash_id,report_date,number_of_rows


In [36]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result2 = con.sql(query2).df()

print("Query 2: March 2026 row count and date range")
display(result2)

Query 2: March 2026 row count and date range


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [37]:
# QUERY 3 — Check GA4 availability

query3 = """
SELECT
    COUNT(*) AS rows_with_ga4_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
"""

result3 = con.sql(query3).df()

print("Query 3: Rows where GA4 data is available")
display(result3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3: Rows where GA4 data is available


,rows_with_ga4_available
0,413966


## Five features

I will use five page-performance features for the March 2026 observation window.

1. **Impressions** — available at the decision moment because the March search impressions have already been observed.
2. **Clicks** — available at the decision moment because the March search clicks have already been observed.
3. **Sessions** — available at the decision moment because the March analytics observations have already been recorded.
4. **CTR** — available at the decision moment because it can be calculated from the observed search impressions and clicks.
5. **Average position** — available at the decision moment because the March search-position observations have already been recorded.

These features describe information available before making the page-review decision. Future outcome information is not used as a feature.


In [38]:
# Build the five-feature frame

query_features = """
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(ga4_sessions) AS sessions,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,

    AVG(gsc_avg_position) AS avg_position

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE ga4_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(query_features).df()

print("Five-feature dataframe:")
display(features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five-feature dataframe:


,client_hash_id,content_hash_id,impressions,clicks,sessions,ctr,avg_position
0,client_65de48885f4ef01b,content_5e120e972f11f833,0.0,0.0,3.0,NaN,NaN
1,client_65de48885f4ef01b,content_4ab81290aec524dd,0.0,0.0,1.0,NaN,NaN
2,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,14.0,0.436681,4.418032
3,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,54.0,0.583312,4.392897
4,client_65de48885f4ef01b,content_aba6e5270431d8ef,0.0,0.0,5.0,NaN,NaN
5,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,30.0,0.688073,8.439390
6,client_65de48885f4ef01b,content_9d17d30b63eaa640,0.0,0.0,1.0,NaN,NaN
7,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,23.0,1.590457,5.531459
8,client_65de48885f4ef01b,content_0535f4407e4320df,0.0,0.0,4.0,NaN,NaN
9,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,2.0,0.000000,2.850000


DATA LEAKAGE


In [41]:
# FOUR — THE TRAP: DELIBERATE DATA LEAKAGE

from sklearn.metrics import accuracy_score

leak_demo = features.copy()

# Create a simple demonstration label
leak_demo["label"] = (
    leak_demo["clicks"] < leak_demo["impressions"] * 0.01
).astype(int)

# DELIBERATELY LEAK THE LABEL INTO ONE FEATURE
leak_demo["leaky_feature"] = leak_demo["label"]

# With leakage, prediction is exactly the label
leaky_prediction = leak_demo["leaky_feature"]

leaky_score = accuracy_score(
    leak_demo["label"],
    leaky_prediction
)

print("Score WITH deliberate leakage:", leaky_score)

# REMOVE THE LEAKED FEATURE
honest_features = leak_demo.drop(
    columns=["leaky_feature"]
)

print("\nLeaky feature removed.")
print("Remaining columns:")
print(honest_features.columns.tolist())

# Confirm that leakage is gone
assert "leaky_feature" not in honest_features.columns

print("\nPASS: The leaked feature is no longer being used.")

Score WITH deliberate leakage: 1.0

Leaky feature removed.
Remaining columns:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'sessions', 'ctr', 'avg_position', 'label']

PASS: The leaked feature is no longer being used.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations.

First, client history is not balanced. Different clients can have different periods of available data, so observations are not equally complete for every client.

Second, GSC and GA4 availability can differ. Some observations may have GSC data while GA4 data is unavailable. Therefore, a zero GA4 value should not automatically be interpreted as zero engagement.

Third, different time windows can overlap when combining tables. A feature must only use information that would have been known before the prediction or ranking decision.

Finally, this is observational data. It can provide measured signals for decision-support and help rank content pages for review, but it cannot prove that refreshing a page will cause traffic or rankings to improve.

The data also cannot tell us or predict Google's internal ranking algorithm.

In [39]:
print("Data limits recorded:")
print("- Client history is not balanced.")
print("- GA4 availability differs across observations.")
print("- Future information must not be used as a feature.")
print("- The data supports decision-support, not causal claims.")

Data limits recorded:
- Client history is not balanced.
- GA4 availability differs across observations.
- Future information must not be used as a feature.
- The data supports decision-support, not causal claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.